In [ ]:
from sklearn.datasets import make_classification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
x, y = make_classification(n_samples=10000, random_state=42)

In [ ]:
df = pd.DataFrame(x)
df["target"] = y

In [ ]:
df = df.drop(columns=[18, 13])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.33, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression()

In [ ]:
model = lr.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)

In [ ]:
log_odds = model.decision_function(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
accuracy_score(y_test, pred)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print(classification_report(y_test, pred))

In [ ]:
confusion_matrix(y_test, pred)

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
result = permutation_importance(
    estimator=model,
    X=X_test,
    y=y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
)

In [ ]:
importance_df = pd.DataFrame(
    {
        "feature": [x for x in range(20)],
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }
)

importance_df = importance_df.sort_values("importance_mean", ascending=False)

In [ ]:
importance_df = importance_df.sort_values("importance_mean")

plt.figure(figsize=(8, 6))

plt.barh(
    importance_df["feature"],
    importance_df["importance_mean"],
    xerr=importance_df["importance_std"],
)

plt.xlabel("Decrease in ROC-AUC after permutation")
plt.ylabel("Feature")
plt.title("Permutation Feature Importance")

plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
cross_val_score(model, X_train, y_train, cv=5)

## Verifying Multicollinearity

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


# ============================================================
# 1. CREATE DATASET
# ============================================================

X, y = make_classification(n_samples=10000, random_state=42)

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 2. CONVERT TO DATAFRAME
# ============================================================

feature_names = [f"X{i}" for i in range(X.shape[1])]

X = pd.DataFrame(X, columns=feature_names)

y = pd.Series(y, name="target")


# ============================================================
# 3. CHECK CORRELATION OF FEATURES 8 AND 13
# ============================================================

print("\nCorrelation between X8 and X13:")

print(X[["X8", "X13"]].corr())


# ============================================================
# 4. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# First:
# 80% -> train + validation
# 20% -> test

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)


# Then:
# 75% of the remaining 80% -> train
# 25% of the remaining 80% -> validation

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=42
)


print("\nDataset sizes:")
print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)


# ============================================================
# 5. CREATE REDUCED FEATURE SET
# ============================================================

# IMPORTANT:
# Here 8 and 13 mean zero-based column indices:
#
# X0, X1, ..., X8, ..., X13, ..., X19

features_to_remove = ["X8", "X13"]

X_train_reduced = X_train.drop(columns=features_to_remove)

X_val_reduced = X_val.drop(columns=features_to_remove)

X_test_reduced = X_test.drop(columns=features_to_remove)


# ============================================================
# 6. CREATE FOUR PIPELINES
# ============================================================

# ------------------------------------------------------------
# Model 1: All features + NO scaling
# ------------------------------------------------------------

logistic_no_scaling = Pipeline([("model", LogisticRegression(max_iter=2000))])


# ------------------------------------------------------------
# Model 2: All features + scaling
# ------------------------------------------------------------

logistic_scaling = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]
)


# ------------------------------------------------------------
# Model 3: Removed X8/X13 + NO scaling
# ------------------------------------------------------------

logistic_reduced_no_scaling = Pipeline([("model", LogisticRegression(max_iter=2000))])


# ------------------------------------------------------------
# Model 4: Removed X8/X13 + scaling
# ------------------------------------------------------------

logistic_reduced_scaling = Pipeline(
    [("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]
)


# ============================================================
# 7. FUNCTION TO CALCULATE METRICS
# ============================================================


def evaluate_model(model, X, y):

    predictions = model.predict(X)

    probabilities = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions),
        "recall": recall_score(y, predictions),
        "f1": f1_score(y, predictions),
        "roc_auc": roc_auc_score(y, probabilities),
    }


# ============================================================
# 8. DEFINE EXPERIMENTS
# ============================================================

experiments = {
    "All features - No scaling": (logistic_no_scaling, X_train, y_train, X_val, y_val),
    "All features - Scaling": (logistic_scaling, X_train, y_train, X_val, y_val),
    "Remove X8,X13 - No scaling": (
        logistic_reduced_no_scaling,
        X_train_reduced,
        y_train,
        X_val_reduced,
        y_val,
    ),
    "Remove X8,X13 - Scaling": (
        logistic_reduced_scaling,
        X_train_reduced,
        y_train,
        X_val_reduced,
        y_val,
    ),
}


# ============================================================
# 9. TRAIN + VALIDATION EVALUATION
# ============================================================

validation_results = []

for name, (model, X_tr, y_tr, X_v, y_v) in experiments.items():
    # Train ONLY on training data
    model.fit(X_tr, y_tr)

    # Evaluate on validation data
    metrics = evaluate_model(model, X_v, y_v)

    metrics["model"] = name

    validation_results.append(metrics)


validation_results = pd.DataFrame(validation_results)

validation_results = validation_results[
    ["model", "accuracy", "precision", "recall", "f1", "roc_auc"]
]

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)

print(
    validation_results.sort_values("accuracy", ascending=False).to_string(index=False)
)


# ============================================================
# 10. SELECT BEST MODEL USING VALIDATION DATA
# ============================================================

best_model_name = validation_results.sort_values("accuracy", ascending=False).iloc[0][
    "model"
]

print("\nBest model according to validation accuracy:")
print(best_model_name)


# ============================================================
# 11. REFIT SELECTED MODEL ON TRAIN + VALIDATION
# ============================================================

# Combine train + validation
X_train_final = pd.concat([X_train, X_val])

y_train_final = pd.concat([y_train, y_val])


# Determine whether selected model needs reduced features

if "Remove X8,X13" in best_model_name:
    X_train_final = X_train_final.drop(columns=features_to_remove)

    X_test_final = X_test.drop(columns=features_to_remove)

else:
    X_test_final = X_test.copy()


# Get the actual selected pipeline
selected_model = dict((name, data[0]) for name, data in experiments.items())[
    best_model_name
]


# ============================================================
# 12. FINAL TRAINING
# ============================================================

selected_model.fit(X_train_final, y_train_final)


# ============================================================
# 13. FINAL TEST EVALUATION
# ============================================================

test_metrics = evaluate_model(selected_model, X_test_final, y_test)


print("\n" + "=" * 80)
print("FINAL TEST RESULTS")
print("=" * 80)

for metric, value in test_metrics.items():
    print(f"{metric:10s}: {value:.4f}")


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    logistic_scaling, X_val, y_val, scoring="roc_auc", n_repeats=20, random_state=42
)

importance = pd.DataFrame(
    {
        "feature": X_val.columns,
        "importance": result.importances_mean,
        "std": result.importances_std,
    }
)

print(importance.sort_values("importance", ascending=False))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Scale X
X_scaled = StandardScaler().fit_transform(X)

# PCA uses ONLY X
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

# Plot
plt.figure(figsize=(10, 7))

for class_value in np.unique(y):
    mask = y == class_value

    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.5, label=f"Class {class_value}")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA: Structure of Input Features")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


# ============================================================
# 1. Train Random Forest
# ============================================================

rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1
)

rf.fit(X_train, y_train)


# ============================================================
# 2. Train Gradient Boosting
# ============================================================

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=10,
    random_state=42,
)

gb.fit(X_train, y_train)


# ============================================================
# 3. Evaluation function
# ============================================================


def evaluate_model(model, X_data, y_data):

    y_pred = model.predict(X_data)
    y_prob = model.predict_proba(X_data)[:, 1]

    return {
        "accuracy": accuracy_score(y_data, y_pred),
        "precision": precision_score(y_data, y_pred),
        "recall": recall_score(y_data, y_pred),
        "f1": f1_score(y_data, y_pred),
        "roc_auc": roc_auc_score(y_data, y_prob),
    }


# ============================================================
# 4. Validation performance
# ============================================================

results = []

for name, model in [("Random Forest", rf), ("Gradient Boosting", gb)]:
    metrics = evaluate_model(model, X_val, y_val)

    results.append({"model": name, **metrics})


results_df = pd.DataFrame(results)

print("\nValidation Performance")
print(results_df.round(4))


In [ ]:
test_metrics = evaluate_model(gb, X_test, y_test)

print("\nFinal Test Performance")
print(pd.Series(test_metrics).round(4))

In [202]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from scipy.stats import randint, uniform

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone


# ============================================================
# 1. Parameter distributions
# ============================================================

param_distributions = {
    "n_estimators": randint(100, 401),
    "learning_rate": uniform(0.01, 0.14),
    "max_depth": randint(2, 6),
    "min_samples_leaf": randint(5, 31),
    "subsample": uniform(0.7, 0.3),
}


# ============================================================
# 2. Generate random parameter combinations
# ============================================================

n_iter = 30
rng = np.random.RandomState(42)

param_list = []

for _ in range(n_iter):
    params = {
        key: distribution.rvs(random_state=rng)
        for key, distribution in param_distributions.items()
    }

    param_list.append(params)


# ============================================================
# 3. Cross-validation setup
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# ============================================================
# 4. Manual randomized search + tqdm
# ============================================================

results = []

base_model = GradientBoostingClassifier(random_state=42)


with tqdm(total=n_iter, desc="Tuning Gradient Boosting", unit="config") as pbar:
    for params in param_list:
        model = clone(base_model)
        model.set_params(**params)

        scores = cross_val_score(
            model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
        )

        results.append(
            {**params, "mean_cv_auc": scores.mean(), "std_cv_auc": scores.std()}
        )

        pbar.update(1)


# ============================================================
# 5. Find best configuration
# ============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values("mean_cv_auc", ascending=False)
    .reset_index(drop=True)
)


best_params = results_df.iloc[0][
    ["n_estimators", "learning_rate", "max_depth", "min_samples_leaf", "subsample"]
].to_dict()


print("\nBest Parameters:")
print(best_params)

print(f"\nBest CV ROC-AUC: {results_df.loc[0, 'mean_cv_auc']:.4f}")


# ============================================================
# 6. Train final model using best parameters
# ============================================================

best_gb = GradientBoostingClassifier(**best_params, random_state=42)

best_gb.fit(X_train, y_train)

Tuning Gradient Boosting:   0%|          | 0/30 [00:00<?, ?config/s]


Best Parameters:
{'n_estimators': 351.0, 'learning_rate': 0.01633182044747533, 'max_depth': 5.0, 'min_samples_leaf': 25.0, 'subsample': 0.8166031869068445}

Best CV ROC-AUC: 0.9718


InvalidParameterError: The 'max_depth' parameter of GradientBoostingClassifier must be an int in the range [1, inf) or None. Got 5.0 instead.

In [203]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score


# ============================================================
# 1. Tuned Gradient Boosting model
# ============================================================

best_gb = GradientBoostingClassifier(
    n_estimators=351,
    learning_rate=0.01633182044747533,
    max_depth=5,
    min_samples_leaf=25,
    subsample=0.8166031869068445,
    random_state=42,
)


# ============================================================
# 2. Train on training data
# ============================================================

best_gb.fit(X_train, y_train)


# ============================================================
# 3. Final test predictions
# ============================================================

y_test_pred = best_gb.predict(X_test)

y_test_proba = best_gb.predict_proba(X_test)[:, 1]


# ============================================================
# 4. Classification report
# ============================================================

print("Final Test Classification Report")
print("=" * 50)

print(classification_report(y_test, y_test_pred, digits=4))


# ============================================================
# 5. ROC-AUC
# ============================================================

test_auc = roc_auc_score(y_test, y_test_proba)

print(f"Test ROC-AUC: {test_auc:.4f}")

Final Test Classification Report
              precision    recall  f1-score   support

           0     0.9497    0.9251    0.9372      1001
           1     0.9268    0.9510    0.9387       999

    accuracy                         0.9380      2000
   macro avg     0.9383    0.9380    0.9380      2000
weighted avg     0.9383    0.9380    0.9380      2000

Test ROC-AUC: 0.9725


In [205]:
# from sklearn.metrics import ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred)

# plt.title("Gradient Boosting — Test Confusion Matrix")
# plt.show()